In [4]:
import pandas as pd
REQUIRED_COLS = ["discharge_MW", "charge_MW", "prices_actual"]


In [5]:
def total_arbitrage_profit(csv_path: str, dt_hours: float, profit_col: str = "profit_step"):
    """
    Loads a CSV, computes profit_step if it doesn't exist, and returns total profit.
    
    profit_step = (discharge_MW - charge_MW) * prices_actual * dt_hours
    """
    df = pd.read_csv(csv_path)

    # 1) Validate required columns
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {csv_path}: {missing}")

    # 2) Make sure numeric columns are numeric (handles strings)
    for c in REQUIRED_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # 3) Compute profit_step only if it doesn't exist
    if profit_col not in df.columns:
        df[profit_col] = (df["discharge_MW"] - df["charge_MW"]) * df["prices_actual"] * dt_hours
    else:
        df[profit_col] = pd.to_numeric(df[profit_col], errors="coerce")

    # 4) Sum (skip NaNs)
    total_profit = df[profit_col].sum(skipna=True)

    return total_profit, df


In [6]:
csv_path = "rlPPO_actual_perfect.csv"   # change this
dt_hours = 1.0              # e.g., 15-min steps = 0.25 hours

total_profit, df = total_arbitrage_profit(csv_path, dt_hours)
print("Total arbitrage profit:", total_profit)


Total arbitrage profit: 1342.7847269367803
